In [51]:
import os
from dotenv import load_dotenv

In [52]:
load_dotenv()
openai_api_key = os.environ["OPENAI_API_KEY"]
#openai_api_key

In [53]:
%cd ..

d:\Agent AI


In [54]:
%cd LC_Project/

d:\Agent AI\LC_Project


In [55]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("data/SDG_goals.pdf")
data = loader.load()

In [56]:
from langchain_text_splitters import TokenTextSplitter
doc_chunks = TokenTextSplitter(
    model_name="gpt-3.5-turbo",
    chunk_size=1000,
    chunk_overlap=100,
    )
chunk_data = doc_chunks.split_documents(data)

In [57]:
type(chunk_data[0])

langchain_core.documents.base.Document

In [58]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo" , temperature=0.3)

In [70]:
prompt_template = """
You are an expert technical interviewer.

Analyze the following document content and generate interview questions
based only on the information provided in the document.

Document:
{text}

Instructions:
- Generate important interview questions from the document.
- Cover important concepts, definitions, processes, goals, and examples.
- Generate a mixture of basic, intermediate, and scenario-based questions.
- Questions must be suitable for an interview.
- Do not create questions about information that is not present in the document.
- Avoid duplicate questions.
- Return only the interview questions.

Interview Questions:
"""

In [71]:
from langchain_core.prompts import PromptTemplate

question_prompt_template = PromptTemplate(
    template=prompt_template,
    input_variables=["text"]
)

In [72]:
refine_template = """
You are an expert technical interviewer.

You already have a list of interview questions generated from the document.

Existing Interview Questions:
{existing_answer}

Additional Document Content:
{text}

Instructions:
- Review the existing interview questions.
- Use the additional document content to generate new relevant questions.
- Add important questions that are missing.
- Remove duplicate or very similar questions.
- Improve unclear questions.
- Keep the questions suitable for interviews.
- Cover important concepts from the additional document content.
- Do not create questions about information that is not present in the document.
- Keep the final list organized and concise.
- Return only the final interview questions.

Final Interview Questions:
"""

In [73]:
refine_prompt_template = PromptTemplate(
    template=refine_template,
    input_variables=["existing_answer", "text"]
)

In [74]:
from langchain_classic.chains.summarize import load_summarize_chain
ques_gen_chain = load_summarize_chain(
    llm = llm , 
    chain_type="refine" , 
    verbose=True , 
    question_prompt=question_prompt_template,
    refine_prompt=refine_prompt_template)

In [75]:
ques = ques_gen_chain.run(chunk_data)
print(ques)



> Entering new RefineDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

You are an expert technical interviewer.

Analyze the following document content and generate interview questions
based only on the information provided in the document.

Document:
4th SDG Y outh Summer Camp – SDG Resource Document 
The 2030 Agenda for Sustainable Development’s 17 
Sustainable Development Goals (SDGs)   
Goal: This document enables 4th SDG Youth Summer Camp participants to i) get to 
know the 17 SDGs, ii) explore what areas each goal covers under its targets, iii) identify 
targets of most interest to participants, and iv) identify synergies between the SDGs 
and chosen target(s).  
 
 
Goal 1. End poverty in all its forms everywhere  
Target 1.1 By 2030, eradicate extreme poverty for all people everywhere, currently measured 
as people living on less than $1.25 a day  
Target 1.2 By 2030, reduce at least by half the proportion of men, women and children of all 


In [76]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [78]:
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(chunk_data, embeddings)

In [84]:
llm_answer_gen = ChatOpenAI(model="gpt-3.5-turbo",temperature=0.1)

In [81]:
ques_list = ques.split("\n")
ques_list

['1. How can a universal, rules-based, open, non-discriminatory, and equitable multilateral trading system under the World Trade Organization be promoted, as per Target 17.10?',
 "2. What strategies can be implemented to significantly increase the exports of developing countries, with a focus on doubling the least developed countries' share of global exports by 2020, as outlined in Target 17.11?",
 '3. How can timely implementation of duty-free and quota-free market access on a lasting basis for all least developed countries be realized, in line with World Trade Organization decisions, as per Target 17.12?',
 '4. What measures can be taken to enhance global macroeconomic stability through policy coordination and coherence, as highlighted in Target 17.13?',
 '5. How can policy coherence for sustainable development be improved, as per Target 17.14?',
 '6. What actions should be taken to enhance the Global Partnership for Sustainable Development and foster multi-stakeholder partnerships t

In [86]:
from langchain_classic.chains import RetrievalQA

ans_generation_chain=RetrievalQA.from_chain_type(
                                                    llm=llm_answer_gen,
                                                    chain_type="stuff",
                                                    retriever=vector_store.as_retriever()
                                                )

In [87]:
for question in ques_list:
    print("Question:" , question)
    answer = ans_generation_chain.run(question)
    print("Answer:" , answer)
    print("...............................\\n\\n")
    # save answer to file
    with open("answer.txt" , "a") as f:
        f.write("Question:" + question + "\\n")
        f.write("Answer:" + answer + "\\n")
        f.write("...........................\\n\\n")

Question: 1. How can a universal, rules-based, open, non-discriminatory, and equitable multilateral trading system under the World Trade Organization be promoted, as per Target 17.10?
Answer: A universal, rules-based, open, non-discriminatory, and equitable multilateral trading system under the World Trade Organization can be promoted by encouraging countries to adhere to the rules and agreements set by the WTO, supporting negotiations under the Doha Development Agenda, and ensuring transparency and fairness in trade practices. Additionally, promoting cooperation and dialogue among member countries to address trade issues and disputes can also contribute to achieving this target.
...............................\n\n
Question: 2. What strategies can be implemented to significantly increase the exports of developing countries, with a focus on doubling the least developed countries' share of global exports by 2020, as outlined in Target 17.11?
Answer: To significantly increase the exports 